<a href="https://colab.research.google.com/github/DharaCS23181/Skincare_Product_Prediction/blob/main/MLP(80_20).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================
# 1. MOUNT DRIVE
# ==============================

from google.colab import drive
drive.mount('/content/drive')


# ==============================
# 2. IMPORT LIBRARIES
# ==============================

import pandas as pd
import numpy as np
import re

from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score

from sklearn.neural_network import MLPClassifier


# ==============================
# 3. LOAD DATA
# ==============================

df = pd.read_csv("/content/drive/MyDrive/ML/skin_recommendation_dataset.csv")

data = df[['skintype','skin_condition','notable_effects','product_type']].copy()


# ==============================
# 4. CLEAN FUNCTION
# ==============================

def clean_text(x):
    x = str(x)
    x = re.sub(r"[^a-zA-Z0-9,\-\s]", "", x)
    return [i.strip().lower() for i in x.split(",") if i.strip() != ""]


data['skintype'] = data['skintype'].apply(clean_text)
data['skin_condition'] = data['skin_condition'].apply(clean_text)
data['notable_effects'] = data['notable_effects'].apply(clean_text)


# ==============================
# 5. ENCODING
# ==============================

mlb_skin = MultiLabelBinarizer()
mlb_condition = MultiLabelBinarizer()
mlb_effects = MultiLabelBinarizer()

skin_features = pd.DataFrame(
    mlb_skin.fit_transform(data['skintype']),
    columns=mlb_skin.classes_
)

condition_features = pd.DataFrame(
    mlb_condition.fit_transform(data['skin_condition']),
    columns=mlb_condition.classes_
)

effects_features = pd.DataFrame(
    mlb_effects.fit_transform(data['notable_effects']),
    columns=mlb_effects.classes_
)

X = pd.concat([skin_features, condition_features, effects_features], axis=1)

le = LabelEncoder()
y = le.fit_transform(data['product_type'])


# ==============================
# 6. TRAIN TEST SPLIT
# ==============================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


# ==============================
# 7. SCALING (IMPORTANT FOR MLP)
# ==============================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# ==============================
# 8. TRAIN MLP MODEL
# ==============================

mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=42
)

mlp.fit(X_train_scaled, y_train)


# ==============================
# 9. EVALUATION
# ==============================

y_pred = mlp.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Weighted F1:", f1_score(y_test, y_pred, average='weighted'))
print(classification_report(y_test, y_pred))


# ==============================
# 10. USER INPUT
# ==============================

print("\nSelect Skin Type:")
for i, val in enumerate(mlb_skin.classes_):
    print(i, ":", val)

skin_choice = int(input("Enter number: "))
user_skin = [mlb_skin.classes_[skin_choice]]


print("\nSelect Skin Condition:")
for i, val in enumerate(mlb_condition.classes_):
    print(i, ":", val)

cond_choice = int(input("Enter number: "))
user_condition = [mlb_condition.classes_[cond_choice]]


print("\nSelect Effect:")
for i, val in enumerate(mlb_effects.classes_):
    print(i, ":", val)

effect_choice = int(input("Enter number: "))
user_effect = [mlb_effects.classes_[effect_choice]]


# ==============================
# 11. CREATE INPUT
# ==============================

user_df = pd.DataFrame(columns=X.columns)
user_df.loc[0] = 0

for val in user_skin + user_condition + user_effect:
    if val in user_df.columns:
        user_df.loc[0, val] = 1

user_input_scaled = scaler.transform(user_df)


# ==============================
# 12. PREDICT
# ==============================

prediction = mlp.predict(user_input_scaled)
predicted_type = le.inverse_transform(prediction)

print("\n✅ Predicted Product Type:", predicted_type[0])


# ==============================
# 13. SMART RECOMMENDATION
# ==============================

filtered = df[df['product_type'] == predicted_type[0]].copy()

filtered['notable_effects'] = filtered['notable_effects'].astype(str).str.lower()
filtered['skin_condition'] = filtered['skin_condition'].astype(str).str.lower()
filtered['skintype'] = filtered['skintype'].astype(str).str.lower()


def calculate_score(row):
    score = 0

    if user_effect[0] in row['notable_effects']:
        score += 3

    if user_condition[0] in row['skin_condition']:
        score += 2

    if user_skin[0] in row['skintype']:
        score += 1

    return score


filtered['score'] = filtered.apply(calculate_score, axis=1)
filtered = filtered.sort_values(by='score', ascending=False)


# ==============================
# 14. TOP 3 RESULTS
# ==============================

print("\n🔥 Top Recommended Products:\n")

top_results = filtered.head(3)

for i, row in top_results.iterrows():
    print("🔹 Product Name:", row['product_name'])
    print("🔹 Brand:", row['brand'])
    print("🔹 Effects:", ", ".join(clean_text(row['notable_effects'])))
    print("🔹 Score:", row['score'])
    print("🔹 Image URL:", row['picture_src'])
    print("-"*40)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Accuracy: 0.4857142857142857
Weighted F1: 0.4795987408722008
              precision    recall  f1-score   support

           0       0.44      0.50      0.47        40
           1       0.37      0.28      0.32        50
           2       0.61      0.63      0.62        62
           3       0.56      0.64      0.60        42
           4       0.38      0.37      0.38        51

    accuracy                           0.49       245
   macro avg       0.47      0.48      0.48       245
weighted avg       0.48      0.49      0.48       245


Select Skin Type:
0 : combination
1 : dry
2 : normal
3 : oily
4 : sensitive
